In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import  roc_auc_score

In [3]:
X_train = pd.read_pickle('../datos/entrenamiento/X_train.pkl')
X_test = pd.read_pickle('../datos/entrenamiento/X_test.pkl')
y_train = pd.read_pickle('../datos/entrenamiento/y_train.pkl')
y_test = pd.read_pickle('../datos/entrenamiento/y_test.pkl')

In [4]:
lgbm = HistGradientBoostingClassifier()

lgbm.fit(X_train, y_train)

lgbm_pred = lgbm.predict_proba(X_test)[:, 1]
print(roc_auc_score(y_test, lgbm_pred))

lgbm_train_pred = lgbm.predict_proba(X_train)[:, 1]
print(roc_auc_score(y_train, lgbm_train_pred))

Exception in thread Thread-4 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\valen\miniconda3\envs\prediccion-riesgo-credito\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\valen\miniconda3\envs\prediccion-riesgo-credito\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\valen\miniconda3\envs\prediccion-riesgo-credito\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\valen\miniconda3\envs\prediccion-riesgo-credito\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa2 in position 106: invalid start byte


0.940124166965873
0.9648018637491323


In [5]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

metricas_cv = cross_val_score(estimator=lgbm, 
                              X=X_train, 
                              y=y_train, 
                              cv=skf, 
                              scoring='roc_auc')
print("ROC AUC Score (Cross-Validation):", metricas_cv.mean())

ROC AUC Score (Cross-Validation): 0.9330387952212547


In [6]:
from sklearn.model_selection import RandomizedSearchCV
grid = {
    'learning_rate': [0.01, 0.05, 0.1],       
    'max_iter': [100, 150, 200],              
    'max_leaf_nodes': [15, 31, 63],            
    'min_samples_leaf': [20, 50, 100],          
    'l2_regularization': [0.0, 1.0, 10.0]       
}
# Configuración del buscador
search = RandomizedSearchCV(
    estimator=lgbm,
    param_distributions=grid,
    n_iter=40,              
    scoring='roc_auc',      
    cv=5,                   
    random_state=42,
    n_jobs=-1              
)

# Ajustar con datos de entrenamiento
search.fit(X_train, y_train)


,estimator,HistGradientB...ngClassifier()
,param_distributions,"{'l2_regularization': [0.0, 1.0, ...], 'learning_rate': [0.01, 0.05, ...], 'max_iter': [100, 150, ...], 'max_leaf_nodes': [15, 31, ...], ...}"
,n_iter,40
,scoring,'roc_auc'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [7]:
print('Mejor combinación:' , search.best_params_)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)
pd.DataFrame(search.cv_results_)[['params', 'mean_test_score']].sort_values('mean_test_score', ascending = False)

Mejor combinación: {'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 10.0}


,params,mean_test_score
4,"{'min_samples_leaf': 50, 'max_leaf_nodes': 31, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 10.0}",0.935478
27,"{'min_samples_leaf': 50, 'max_leaf_nodes': 15, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 0.0}",0.935353
7,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_iter': 150, 'learning_rate': 0.1, 'l2_regularization': 1.0}",0.934852
39,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 10.0}",0.934346
29,"{'min_samples_leaf': 100, 'max_leaf_nodes': 15, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 1.0}",0.934157
33,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_iter': 100, 'learning_rate': 0.1, 'l2_regularization': 0.0}",0.934035
3,"{'min_samples_leaf': 50, 'max_leaf_nodes': 63, 'max_iter': 150, 'learning_rate': 0.1, 'l2_regularization': 10.0}",0.933753
15,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63, 'max_iter': 200, 'learning_rate': 0.1, 'l2_regularization': 1.0}",0.933707
2,"{'min_samples_leaf': 100, 'max_leaf_nodes': 63, 'max_iter': 150, 'learning_rate': 0.1, 'l2_regularization': 1.0}",0.933641
20,"{'min_samples_leaf': 20, 'max_leaf_nodes': 63, 'max_iter': 100, 'learning_rate': 0.05, 'l2_regularization': 0.0}",0.933517
